# 10 · Model comparison — compare models under the same conditions

**The question:** did a trained model improve on the base, and how does it compare with GPT-5.4 and Gemini?

Edit **`configs/model_comparison.yaml`**, then run this notebook from the top. It uses notebook 02's three prompt conditions and the same development questions/scorer. The initial selection is **base + CPT**, with generation switched off until you set `execution.run: true`. Add `openai_54` and `gemini` to `selected_models` to include paid APIs. The preview shows request counts before any inference.

All loading, generation and scoring code is visible below. This notebook performs inference only; it does not train or merge your saved files. Start with a fresh kernel and release model-holding kernels from earlier notebooks if GPU memory is tight.

**CPT → SFT means a training history.** If SFT continued the same CPT adapter, load the final adapter once. If CPT was merged into a new base before SFT, load that merged base plus its SFT adapter. Two independently trained adapters are not a substitute for that history. Notebook 05 currently starts from the original base.

In [ ]:
# Prepare a configurable comparison without loading a model or contacting a provider.
# Path handles files; sys controls imports; os reads environment variables; json serializes records.
# hashlib creates reproducibility fingerprints; time measures calls; gc releases unused Python objects;
# warnings captures loader warnings. datetime/timezone supply timestamps; Counter counts statuses.
# importlib.metadata reads installed package versions; yaml reads settings; pandas and display show tables.

from pathlib import Path
import sys, os, json, hashlib, time, gc, warnings
from datetime import datetime, timezone
from collections import Counter
import importlib.metadata as metadata
import yaml
import pandas as pd
from IPython.display import display

# Works when the notebook starts in the project root or notebooks/.
# Walk the current directory and its parents until src/lab.py identifies the project root.
# `*Path.cwd().parents` unpacks ancestor paths into the list; next(...) selects the first match.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/lab.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import lab

# safe_load turns YAML into nested dictionaries without executing YAML-specified Python objects.
# PLAY controls this comparison; CFG retains the shared lab's model/device paths.
CONFIG_PATH = ROOT / "configs/model_comparison.yaml"
PLAY = yaml.safe_load(CONFIG_PATH.read_text())
CFG = lab.cfg()
EXEC = PLAY["execution"]
EV = PLAY["evaluation"]
SELECTED = PLAY["selected_models"]
REGISTRY = PLAY["models"]
# assert rejects invalid settings early. set(...) removes duplicates, so comparing lengths
# catches repeated model IDs; all(...) checks every selected item against its allowed values.
assert isinstance(EXEC["run"], bool), "execution.run must be true or false"
assert isinstance(EXEC["retry_failed"], bool)
assert isinstance(EXEC["max_api_calls"], int) and EXEC["max_api_calls"] >= 0
assert SELECTED and len(SELECTED) == len(set(SELECTED)), "Select unique model IDs"
assert all(k in REGISTRY for k in SELECTED), "Unknown model ID in selected_models"
assert EV["conditions"] and len(EV["conditions"]) == len(set(EV["conditions"]))
assert all(c in ("naive", "engineered", "with_docs") for c in EV["conditions"])
assert isinstance(EV["local_max_new_tokens"], int) and EV["local_max_new_tokens"] > 0
assert isinstance(EV["local_max_input_tokens"], int) and EV["local_max_input_tokens"] > 0


In [ ]:
# Define three small helpers for reproducible keys, file hashes, and project-relative paths.
# Defining a function stores its instructions; its body runs only when another cell calls it.


# Make a stable key for JSON-compatible settings: sort_keys removes dictionary-order differences,
# encode converts text to bytes, and SHA-256 produces a hexadecimal fingerprint.
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

# Read file bytes through a context manager so the stream closes even on failure.
# The 'rb' mode is binary, allowing model/config files to be hashed without text conversion.
def file_sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

# expanduser resolves '~'; is_absolute distinguishes full paths from project-relative paths.
# The conditional expression chooses the appropriate base, and resolve normalizes the result.
def project_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else ROOT / path).resolve()


In [ ]:
# Record the installed library versions and display the selected settings.
# Version metadata is read locally; this cell does not import a model checkpoint or start an API client.


# Ask package metadata for version strings without importing/loading each model runtime.
# A missing package is recorded as text so preflight can skip unsupported candidates later.
VERSIONS = {}
for name in ["torch", "transformers", "peft", "openai", "google-genai"]:
    try:
        VERSIONS[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        VERSIONS[name] = "not installed"

print("Config:", CONFIG_PATH)
print("Selected models:", SELECTED)
print("Generation enabled:", EXEC["run"])
print("Installed libraries:", VERSIONS)


## Choose the questions and prompt conditions

These system prompts reproduce notebook 02. **Naive** asks without special instructions; **engineered** asks for grounded answers but supplies no documents; **with_docs** supplies the same complete local document collection.

Use the same condition when comparing a trained model to its base. A base with documents versus CPT without documents changes two things at once. GPT-5.4/Gemini have not learned this fictional company in our experiment, so their with-documents results are the useful test of reading the provided facts.

`max_questions: null` uses all 36 development questions. A smaller number is a quick trial on the first N questions, not a replacement benchmark. Notebook 08 retains the final held-out evaluation. `custom_question` optionally adds one unscored playground question per condition/model.

In [ ]:
# Build three prompt conditions and choose the dev questions for every selected model.
# Naive changes only the system instruction; engineered supplies a grounding rule; with_docs also supplies sources.
# Keep the same condition when comparing model weights so document access is not a hidden difference.

NAIVE = "You are a helpful assistant."
ENGINEERED = (
    "You are the customer assistant for Meridian Motors, an electric two-wheeler company. "
    "Answer only from Meridian's official documents. If the documents do not contain the "
    "answer, say plainly that you do not have that information and offer to connect the "
    "customer to a dealer. Never guess a number. Never round a range figure upward."
)
# glob finds Markdown source files, sorted fixes their order, and join separates them with blank lines.
# This reads every matching raw document; it does not perform retrieval.
DOCS = "\n\n".join(p.read_text() for p in sorted(lab.RAW.glob("*.md")))
SYSTEMS = {
    "naive": NAIVE,
    "engineered": ENGINEERED,
    "with_docs": ENGINEERED + "\n\n=== MERIDIAN OFFICIAL DOCUMENTS ===\n" + DOCS,
}
all_dev = lab.eval_set("dev")
limit = EV.get("max_questions")
assert limit is None or (isinstance(limit, int) and not isinstance(limit, bool) and 0 < limit <= len(all_dev))
# Use all dev questions when the limit is None, otherwise the first N via [:limit].
# A small prefix is a smoke sample, not a random or representative quality benchmark.
items = all_dev if limit is None else all_dev[:limit]
assert len({q["id"] for q in items}) == len(items)
fidx = lab.fact_index()
# A dictionary comprehension builds ID -> question lookup for later scoring.
# The custom question uses answerable=None to mark it as a playground prompt without a benchmark label.
items_by_id = {q["id"]: q for q in items}
custom = EV.get("custom_question")
assert custom is None or (isinstance(custom, str) and custom.strip())
questions = items + ([{"id": "custom", "question": custom, "answerable": None}] if custom else [])

# Booleans add as 1/0, so sum(q['answerable'] ...) counts answerable questions.
# DataFrame(...)[[...]].head(5) previews selected columns for the first five rows.
print(f"Dev questions: {len(items)} ({sum(q['answerable'] for q in items)} answerable)")
print("Prompt conditions:", EV["conditions"])
print("Optional custom question:", custom)
display(pd.DataFrame(items)[["id", "question", "answerable"]].head(5))


## Check candidate files before loading anything

`local_base` and `local_merged` require a complete Hugging Face causal-model checkpoint. `local_adapter` requires a complete base plus **one** PEFT LoRA adapter. Paths can be absolute or relative to the project root. A null base path uses the original model path in `configs/config.yaml`; a null tokenizer path uses that base.

A missing or incompatible candidate is marked **SKIPPED**, never replaced with the base. The adapter's recorded base must agree with the configured base. This catches common mistakes, but paths alone cannot prove training history; keep the training run and base weights that produced the adapter. MLX/GGUF checkpoints require a different runtime and are not accepted by this PyTorch loader.

In [ ]:
# Validate registry candidates and describe their artifact identity without loading model weights.
# A full checkpoint is different from a LoRA adapter: the latter still needs its matching base.
# The READY dictionary holds usable candidates; INVENTORY keeps both ready and skipped outcomes for display.
# These definitions stay together because another notebook's authoring script copies this complete preflight cell.

LOCAL_KINDS = {"local_base", "local_adapter", "local_merged"}
API_KINDS = {"openai", "gemini"}

# Accept a complete single weight file or all shards named by an index JSON file.
# A sharded checkpoint stores tensors across several files; every listed shard must exist.
def checkpoint_files(path):
    if not (path / "config.json").is_file():
        raise ValueError(f"No complete model config at {path}")
    if (path / "adapter_config.json").exists():
        raise ValueError(f"Adapter directory supplied as a full base: {path}")
    single = [path / n for n in ("model.safetensors", "pytorch_model.bin") if (path / n).is_file()]
    if single:
        return single
    for name in ("model.safetensors.index.json", "pytorch_model.bin.index.json"):
        index = path / name
        if index.is_file():
            # weight_map values name shard files; set removes repeated names and sorted fixes their order.
            # [index, *shards] later unpacks all shard paths into one returned list.
            shards = [path / n for n in sorted(set(json.loads(index.read_text())["weight_map"].values()))]
            if shards and all(p.is_file() for p in shards):
                return [index, *shards]
    raise ValueError(f"No complete model weights at {path}")

# Copy the supplied dictionary so resolved paths do not mutate the original registry entry.
# LOCAL_KINDS | API_KINDS is set union: a kind may belong to either allowed group.
def inspect_candidate(model_id, supplied):
    spec = dict(supplied)
    kind = spec["kind"]
    if kind not in LOCAL_KINDS | API_KINDS:
        raise ValueError(f"Unsupported kind: {kind}")
    # Reject unsupported combinations rather than silently ignoring them or stacking unrelated adapters.
    # A null/absent optional value is read safely with .get(); mandatory fields use square brackets.
    if "adapters" in spec or "adapter_paths" in spec:
        raise ValueError("Use one final adapter and its correct base; adapter lists are unsupported")
    if kind != "local_adapter" and spec.get("adapter_path") is not None:
        raise ValueError("adapter_path requires kind: local_adapter; it must not be silently ignored")
    # API candidates need an installed SDK and model/key-setting names, but no key is read here.
    # For Gemini, google.genai.types validates the optional thinking configuration locally.
    if kind in API_KINDS:
        package = "openai" if kind == "openai" else "google-genai"
        if VERSIONS[package] == "not installed":
            raise ValueError(f"Install {package} in this notebook's kernel first")
        assert spec.get("model") and spec.get("api_key_name")
        assert isinstance(spec["max_output_tokens"], int) and spec["max_output_tokens"] > 0
        assert spec.get("timeout_seconds", 90) > 0
        if kind == "gemini" and spec.get("thinking_config") is not None:
            from google.genai import types
            # **spec['thinking_config'] expands dictionary entries into named constructor arguments.
            # This checks their schema; it does not call the model.
            types.ThinkingConfig(**spec["thinking_config"])  # local schema validation only
        return spec, {"provider": kind, "requested_model": spec["model"]}

    # Use an explicitly configured base/tokenizer when present; otherwise fall back to shared base settings.
    # This fallback concerns an optional base path, not substituting a missing requested adapter.
    base = project_path(spec.get("base_path") or CFG["model"]["base_path"])
    token_path = project_path(spec.get("tokenizer_path") or str(base))
    weights = checkpoint_files(base)
    if not (token_path / "tokenizer_config.json").is_file():
        raise ValueError(f"Missing tokenizer config at {token_path}")
    spec.update(base_path=str(base), tokenizer_path=str(token_path))
    # Large base weights use file metadata; adapter files and model/tokenizer
    # configs are content-hashed. Change cache_tag after replacing a base while
    # deliberately preserving its file size and modification time.
    # This existing fingerprint hashes small config/tokenizer files but identifies base weight files
    # by path, size and modification time. Replacing weights while preserving those can defeat cache invalidation;
    # the cache_tag is the existing manual way to force a new comparison in that case.
    fingerprint = {
        "base": str(base),
        "base_weight_files": [{"path": str(p), "bytes": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns} for p in weights],
        "config_hashes": {str(p): file_sha(p) for p in sorted(set(
            [*base.glob("*.json"), *token_path.glob("*.json"), *token_path.glob("*.jinja"),
             *token_path.glob("tokenizer.*"), *token_path.glob("*.model"), *token_path.glob("*.spm"),
             *token_path.glob("*.txt"), *token_path.glob("*.tiktoken"), *token_path.glob("*.bpe")])) if p.is_file()},
    }
    # Require both an adapter configuration and a weight file. next(generator, None) returns the first
    # existing format or None when neither exists, allowing a clear skip message.
    if kind == "local_adapter":
        adapter = project_path(spec["adapter_path"])
        config_file = adapter / "adapter_config.json"
        weight_file = next((adapter / n for n in ("adapter_model.safetensors", "adapter_model.bin") if (adapter / n).is_file()), None)
        if not config_file.is_file() or weight_file is None:
            raise ValueError(f"No saved adapter config + weights at {adapter}; finish/save training first")
        adapter_cfg = json.loads(config_file.read_text())
        if adapter_cfg.get("task_type") != "CAUSAL_LM" or adapter_cfg.get("peft_type") != "LORA":
            raise ValueError("This loader expects a CAUSAL_LM LoRA adapter")
        # Verify the adapter's recorded base path agrees with the selected base path.
        # Path agreement is a compatibility check, not complete proof of an adapter's training history.
        recorded_base = adapter_cfg.get("base_model_name_or_path")
        if not recorded_base or project_path(recorded_base) != base:
            raise ValueError(f"Adapter records base {recorded_base!r}, but config selects {str(base)!r}")
        spec["adapter_path"] = str(adapter)
        fingerprint["adapter"] = {"config": file_sha(config_file), "weights": file_sha(weight_file), "recorded_base": recorded_base}
    return spec, fingerprint

# Try each selected candidate. A validation exception marks it SKIPPED with its reason;
# there is no base-model fallback when the requested adapter is missing.
READY, INVENTORY = {}, []
for model_id in SELECTED:
    try:
        spec, artifact = inspect_candidate(model_id, REGISTRY[model_id])
        READY[model_id] = {"spec": spec, "artifact": artifact}
        status = "READY (key checked only if run)" if spec["kind"] in API_KINDS else "READY"
        detail = spec.get("adapter_path") or spec.get("base_path") or spec.get("model")
    except (ValueError, OSError, KeyError, AssertionError) as exc:
        status, detail = "SKIPPED", str(exc)
    INVENTORY.append({"model": model_id, "label": REGISTRY[model_id].get("label", model_id), "status": status, "detail": detail})
display(pd.DataFrame(INVENTORY))


## Preview requests and reuse exact cached requests

Each model answers each question separately, with no conversation history. Results are cached by model artifact/config, prompt, question, library versions and `cache_tag`. Rerunning reuses cached records. Changed adapter weights create new cache keys. Changing `cache_tag` forces a fresh experiment, including paid calls if selected. A `latest` API alias may change upstream without a local fingerprint change; use a new cache tag to retest it, and inspect the returned model version.

Failed records stay visible. Set `retry_failed: true` only when you want to retry them; automatic SDK retries are disabled. `max_api_calls` limits this execution's **uncached requests**, not currency spend. API timeout/failed calls can still be billable. API credentials are read only when execution starts, from the named environment variable or the existing secrets file, and never printed or saved.

In [ ]:
# Load prior response records and preview the requests this selection would require.
# JSONL means one JSON object per line, useful for appending each completed request independently.
# The cache prevents repeat work only when its request key matches; a replay is not a new trial.

CACHE_DIR = lab.RUNS / "model_comparison"
CACHE_FILE = CACHE_DIR / "responses.jsonl"
CACHE = {}
# enumerate(...,1) supplies human-friendly line numbers. Blank lines are skipped;
# a malformed record raises an error rather than silently dropping part of the experiment.
if CACHE_FILE.is_file():
    for line_no, line in enumerate(CACHE_FILE.read_text().splitlines(), 1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
            CACHE[record["cache_key"]] = record  # newest attempt for the same key wins
        except (json.JSONDecodeError, KeyError):
            raise ValueError(f"Incomplete cache line {line_no}; repair it before resuming: {CACHE_FILE}")

# Hash model identity, prompt condition, actual question text, token limits, versions, and device.
# A changed cache_tag deliberately produces new keys and can therefore cause new paid requests.
def request_key(model_id, candidate, condition, question):
    return digest({"implementation": "model-comparison-v1", "cache_tag": EXEC["cache_tag"],
                   "model_id": model_id, "spec": candidate["spec"], "artifact": candidate["artifact"],
                   "system": SYSTEMS[condition], "question": question["question"],
                   "local_limits": [EV["local_max_new_tokens"], EV["local_max_input_tokens"]],
                   "versions": VERSIONS, "device": CFG["device"]["torch"]})

# Request again if no cache exists, or if retry_failed is enabled and the stored status is not ok.
# `or` and `and` short-circuit, so cached['status'] is read only when a cached record exists.
def should_request(key):
    cached = CACHE.get(key)
    return cached is None or (EXEC["retry_failed"] and cached["status"] != "ok")


In [ ]:
# Build and display the model/condition/question request plan from the prepared cache.
# This cell counts new and reused requests. Generation remains controlled by EXEC['run'] later.


# Build one job for every ready model × condition × question combination.
# The nested loops define the experiment grid but do not execute generation here.
JOBS, plan = [], []
for model_id, candidate in READY.items():
    current = []
    for condition in EV["conditions"]:
        for q in questions:
            job = {"model_id": model_id, "condition": condition, "question": q,
                   "cache_key": request_key(model_id, candidate, condition, q)}
            JOBS.append(job)
            current.append(job)
    # Summing boolean request decisions counts new work. Paid request counts include only API kinds;
    # local model requests still cost computation, but they are not provider API calls.
    uncached = sum(should_request(j["cache_key"]) for j in current)
    plan.append({"model": model_id, "questions_per_condition": len(questions),
                 "conditions": len(EV["conditions"]), "total": len(current),
                 "new_requests": uncached, "cached": len(current)-uncached,
                 "paid_api_requests": uncached if candidate["spec"]["kind"] in API_KINDS else 0})
planned_api_calls = sum(row["paid_api_requests"] for row in plan)
display(pd.DataFrame(plan))
print(f"Planned new API calls: {planned_api_calls}; configured cap: {EXEC['max_api_calls']}")
print("Cache:", CACHE_FILE)
print("Local output cap:", EV["local_max_new_tokens"], "tokens; provider caps are in each model's config.")
if not EXEC["run"]:
    print("PREVIEW ONLY. Set execution.run: true in the YAML and rerun from the top to generate.")


## Local model generation: base plus the selected final adapter

Load a fresh base for each candidate because attaching PEFT modifies that object. After one candidate finishes, release it before loading the next. This keeps the baseline separate and reduces memory use.

Greedy decoding, the chat template and the default 160-token output cap match notebook 02. The tokenizer must fit the model vocabulary; we do not resize embeddings. Inputs over the configured 3,072-token cap are reported instead of silently losing part of the documents. A generated answer that reaches its output cap without an EOS is recorded as truncated.

In [ ]:
# Define local loading and answering behavior; no weights are loaded until load_local is called.
# torch provides tensors and device execution; transformers loads the causal model and tokenizer.
# PEFT's PeftModel attaches one saved LoRA adapter. The function names describe different phases:
# load_local prepares a runtime; local_answer uses it for one request.

def load_local(spec):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    # Read the configured device and fail if Apple MPS was requested but is unavailable.
    # local_files_only avoids checkpoint downloads; trust_remote_code=False avoids custom repository code.
    device = CFG["device"]["torch"]
    if device == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS unavailable; choose cpu in configs/config.yaml if intended")
    tokenizer = AutoTokenizer.from_pretrained(spec["tokenizer_path"], local_files_only=True, trust_remote_code=False)
    if not tokenizer.chat_template:
        raise ValueError("Selected tokenizer has no chat template")
    # A tokenizer needs a padding ID for generation. Reusing EOS as padding is the chosen fallback;
    # if neither exists, stop rather than inventing a vocabulary token.
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.pad_token_id is None:
        raise ValueError("Tokenizer needs a pad or EOS token")
    mdl = AutoModelForCausalLM.from_pretrained(
        spec["base_path"], dtype=torch.float32, local_files_only=True, trust_remote_code=False)
    # Attach the adapter to a fresh base. is_trainable=False keeps this an inference candidate.
    # catch_warnings records loader warnings temporarily so missing adapter tensors can be rejected.
    if spec["kind"] == "local_adapter":
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            mdl = PeftModel.from_pretrained(mdl, spec["adapter_path"], is_trainable=False, local_files_only=True)
        if any("missing adapter keys" in str(w.message).lower() for w in caught):
            raise ValueError("Adapter has missing tensors; refusing to evaluate an incomplete adapter")
        for warning in caught:
            warnings.warn(str(warning.message), warning.category)
    # Check that every tokenizer ID can index both model input and output vocabularies.
    # Vocabulary row count is not the same as hidden size: it counts possible token IDs.
    input_rows = mdl.get_input_embeddings().weight.shape[0]
    output_rows = mdl.get_output_embeddings().weight.shape[0]
    if max(tokenizer.get_vocab().values()) >= min(input_rows, output_rows):
        raise ValueError("Tokenizer IDs exceed the model vocabulary; use its training tokenizer")
    # Disable parameter gradients and switch the model to eval mode. The underscore means in-place change.
    # .to(device) moves the model; eval() changes module behavior but is distinct from no-grad execution.
    mdl.requires_grad_(False)
    mdl.to(device).eval()
    return {"model": mdl, "tokenizer": tokenizer, "device": device}

# Use one model/tokenizer runtime for a single fresh system+user request, with no prior conversation.
# The chat template determines special role tokens before ordinary text tokenization.
def local_answer(runtime, system, question):
    import torch
    mdl, tokenizer, device = runtime["model"], runtime["tokenizer"], runtime["device"]
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    # input_ids has [batch, sequence] shape; index 1 is sequence length.
    # Inputs over the cap return a status without generation, rather than silently dropping document text.
    n_input = encoded["input_ids"].shape[1]
    if n_input > EV["local_max_input_tokens"]:
        return {"status": "input_too_long", "answer": "", "input_tokens": n_input,
                "finish_reason": "Input exceeds configured cap; no generation performed"}
    encoded = encoded.to(device)
    # Wait for queued GPU work before timing so asynchronous execution does not understate duration.
    if device == "mps": torch.mps.synchronize()
    if str(device).startswith("cuda"): torch.cuda.synchronize()
    t0 = time.perf_counter()
    # This context disables autograd bookkeeping for inference. **encoded expands named input tensors.
    # do_sample=False and num_beams=1 select greedy decoding; max_new_tokens counts only generated tokens.
    with torch.inference_mode():
        output = mdl.generate(**encoded, max_new_tokens=EV["local_max_new_tokens"],
                              do_sample=False, num_beams=1, num_return_sequences=1,
                              return_dict_in_generate=False, pad_token_id=tokenizer.eos_token_id)
    if device == "mps": torch.mps.synchronize()
    if str(device).startswith("cuda"): torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    # The generated tensor includes the original prompt. [0][n_input:] keeps only the first answer's suffix.
    # .tolist() brings token IDs into an ordinary Python list for EOS inspection and decoding.
    new_ids = output[0][n_input:].tolist()
    # EOS may be one ID or a list; convert either representation into a set for membership testing.
    # An answer that reaches the cap without EOS is labelled truncated and is not a completed quality sample.
    eos = mdl.generation_config.eos_token_id
    eos_ids = set(eos if isinstance(eos, list) else [eos])
    ended = bool(new_ids and new_ids[-1] in eos_ids)
    answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    status = "ok" if answer and ended else "truncated" if len(new_ids) >= EV["local_max_new_tokens"] else "empty"
    return {"status": status, "answer": answer, "finish_reason": "eos" if ended else status,
            "input_tokens": n_input, "output_tokens": len(new_ids), "elapsed_seconds": elapsed,
            "generation_config": mdl.generation_config.to_dict()}


## OpenAI GPT-5.4 and Gemini generation

GPT-5.4 uses the Responses API with reasoning effort `none` and temperature `0` by default. Higher reasoning settings omit temperature because that combination is not supported. Gemini uses the configured model alias and temperature `0`; model-specific thinking settings are optional.

The default caps are **Qwen 160, GPT-5.4 160, Gemini 1,024**. Gemini's budget can include thinking, so these are **not equal-compute limits**, and tokenizers differ. The comparison keeps questions and source text aligned; usage/latency are recorded separately. There is no web search, retrieval tool or prior conversation added to API requests. Temperature zero does not guarantee byte-identical API responses.

Only completed, nonempty outputs are scored. Blocked/empty/truncated/API-error results remain visible but cannot become a misleading zero accuracy or a successful refusal. On the first unsuccessful API response, the runner stops that candidate's remaining new requests so you can inspect the settings before spending more.

References: [GPT-5.4 guidance](https://developers.openai.com/api/docs/guides/latest-model?model=gpt-5.4), [Responses API](https://developers.openai.com/api/reference/python/resources/responses/methods/create), [Gemini thinking](https://ai.google.dev/gemini-api/docs/generate-content/thinking), [Gemini model versions](https://ai.google.dev/gemini-api/docs/models#model-version-name-patterns).

In [ ]:
# Define optional provider clients and response parsers; calling these functions may contact APIs.
# os reads environment variables, while lab.load_secret is the existing private fallback for named keys.
# The OpenAI SDK sends Responses requests; google.genai sends Gemini content-generation requests.
# These definitions stay together because the notebook-12 authoring script copies the provider cell.

def read_api_key(spec):
    name = spec["api_key_name"]
    # Prefer a named environment variable. The secret value is returned in memory, not printed.
    # Raising from None hides the original exception chain when reporting a missing key name.
    value = os.environ.get(name)
    if value:
        return value
    try:
        return lab.load_secret(name)
    except (FileNotFoundError, KeyError):
        raise RuntimeError(f"Missing {name}: set the environment variable or add it to the existing secrets file") from None

# Construct the selected SDK client only when execution needs it.
# max_retries=0 / attempts=1 disables automatic retry loops that could multiply paid requests.
def create_api_client(spec):
    key = read_api_key(spec)
    if spec["kind"] == "openai":
        from openai import OpenAI
        return OpenAI(api_key=key, max_retries=0, timeout=spec.get("timeout_seconds", 90))
    from google import genai
    from google.genai import types
    # Gemini's timeout uses milliseconds, so seconds are multiplied by 1000.
    # types.HttpOptions and HttpRetryOptions are typed settings containers, not model responses.
    return genai.Client(api_key=key, http_options=types.HttpOptions(
        timeout=int(spec.get("timeout_seconds", 90)*1000),
        retry_options=types.HttpRetryOptions(attempts=1)))

# Build an OpenAI request with the chosen model, system instructions, user question and output budget.
# kwargs is an ordinary dictionary; **kwargs expands it into named API method arguments.
def openai_answer(client, spec, system, question):
    effort = spec.get("reasoning_effort", "none")
    kwargs = {"model": spec["model"], "instructions": system,
              "input": [{"role": "user", "content": question}],
              "reasoning": {"effort": effort}, "max_output_tokens": spec["max_output_tokens"],
              "store": False}
    # Temperature zero is added only for the configured no-reasoning setting in this implementation.
    # store=False requests that the response not be stored for later retrieval through this API setting.
    if effort == "none": kwargs["temperature"] = 0.0
    t0 = time.perf_counter()
    response = client.responses.create(**kwargs)
    answer = response.output_text.strip()
    # getattr(object, name, default) safely handles optional fields absent from a response.
    # The nested generator checks every output content part for a refusal marker.
    details = getattr(response, "incomplete_details", None)
    reason = getattr(details, "reason", None)
    refusal = any(getattr(part, "type", None) == "refusal"
                  for output in response.output for part in (getattr(output, "content", None) or []))
    # The existing conditional chain labels completion, truncation, refusal, or other incomplete states.
    # It checks nonempty completed output before refusal; a mixed text+refusal response could therefore be marked ok.
    # That ordering is an existing behavior, documented here without changing the executable code.
    status = ("ok" if response.status == "completed" and answer else
              "truncated" if reason == "max_output_tokens" else
              "blocked" if refusal else "empty" if response.status == "completed" else "incomplete")
    # model_dump(mode='json') turns the SDK's typed usage object into a serializable dictionary.
    # Provider-reported input/output counts are saved, along with returned model and response IDs.
    usage = response.usage.model_dump(mode="json") if response.usage else {}
    return {"status": status, "answer": answer, "finish_reason": reason or response.status,
            "elapsed_seconds": time.perf_counter()-t0, "usage": usage,
            "input_tokens": usage.get("input_tokens"), "output_tokens": usage.get("output_tokens"),
            "resolved_model": response.model, "response_id": response.id}

# Build Gemini's provider-specific settings, including optional thinking configuration.
# The system instruction and question remain separate; no search tool or prior conversation is supplied.
def gemini_answer(client, spec, system, question):
    from google.genai import types
    settings = {"system_instruction": system, "temperature": 0.0,
                "max_output_tokens": spec["max_output_tokens"]}
    if spec.get("thinking_config") is not None:
        settings["thinking_config"] = types.ThinkingConfig(**spec["thinking_config"])
    t0 = time.perf_counter()
    response = client.models.generate_content(model=spec["model"], contents=question,
                                               config=types.GenerateContentConfig(**settings))
    # Use the first returned candidate when present; safely handle a response with none.
    # A finish-reason enum may expose .value; the fallback converts an ordinary object to text.
    candidate = response.candidates[0] if response.candidates else None
    finish = getattr(candidate, "finish_reason", None)
    finish_name = getattr(finish, "value", str(finish) if finish else None)
    content = getattr(candidate, "content", None)
    # Concatenate text parts while excluding thought parts. This extracts visible answer text,
    # not hidden reasoning. A successful finish still needs a nonempty answer to earn status ok.
    answer = "".join(part.text for part in (getattr(content, "parts", None) or [])
                     if getattr(part, "text", None) and not getattr(part, "thought", False)).strip()
    status = ("ok" if finish_name == "STOP" and answer else
              "truncated" if finish_name == "MAX_TOKENS" else
              "empty" if finish_name == "STOP" else "blocked_or_incomplete")
    usage = response.usage_metadata.model_dump(mode="json") if response.usage_metadata else {}
    return {"status": status, "answer": answer, "finish_reason": finish_name,
            "elapsed_seconds": time.perf_counter()-t0, "usage": usage,
            "input_tokens": usage.get("prompt_token_count"), "output_tokens": usage.get("candidates_token_count"),
            "thinking_tokens": usage.get("thoughts_token_count"),
            "resolved_model": getattr(response, "model_version", None),
            "response_id": getattr(response, "response_id", None)}


## Run the selected comparison

The YAML's `execution.run` switch controls this cell. It loads one local model at a time, saves each response immediately, and reuses matching cache records. `execution.max_api_calls` is checked before inference. After API errors, fix the configuration/key or output budget, then rerun; failed entries are retried only when `retry_failed` is true or their request key changes.

An interrupted/partial run is useful for inspecting answers. It does not earn a complete score. No generation happens while `run` is false, but the cells after this can still display historical results and any existing cache.

In [ ]:
# Execute the planned comparison only when its run switch is enabled, caching each result immediately.
# Local candidates are loaded one at a time; provider clients are opened only for selected API candidates.
# The error helper avoids saving raw SDK exceptions, which may contain request metadata.
# The finally block releases resources even if setup or generation fails.

# getattr reads optional provider error fields; isinstance restricts the saved code to simple values.
# The error class is retained for debugging without serializing the whole exception.
def safe_error(exc):
    # Do not log raw SDK exceptions: they can contain request contents/headers.
    code_value = getattr(exc, "status_code", None) or getattr(exc, "code", None)
    return {"status": "error", "answer": "", "error_type": type(exc).__name__,
            "error_code": code_value if isinstance(code_value, (str, int)) else None}

# Combine request identity and result fields in one dictionary; **result unpacks the result entries.
# A UTC ISO timestamp makes the saved record unambiguous across time zones.
def save_record(job, candidate, result):
    record = {"cache_key": job["cache_key"], "model_id": job["model_id"],
              "condition": job["condition"], "question_id": job["question"]["id"],
              "question": job["question"]["question"], "model_spec": candidate["spec"],
              "artifact": candidate["artifact"], "versions": VERSIONS,
              "system_prompt": SYSTEMS[job["condition"]],
              "created_at": datetime.now(timezone.utc).isoformat(), **result}
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    # Open in append mode ('a') so previous cache lines remain. The with block closes the file safely;
    # flush sends buffered text to the file before the next request begins.
    with CACHE_FILE.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        stream.flush()
    CACHE[job["cache_key"]] = record
    return record


In [ ]:
# Run the pending jobs under the configured execution switch and API request cap.
# This is the cell that actually loads models or calls providers; the previous helpers only defined behavior.
# Completed and failed attempts are appended to the cache as they occur.


# Reset per-execution accounting. The cap counts uncached API attempts, including failed attempts.
# If the whole plan exceeds the cap, stop before loading any candidate.
api_calls_this_execution = 0
run_notes = []
if EXEC["run"]:
    if planned_api_calls > EXEC["max_api_calls"]:
        raise ValueError(f"Plan requires {planned_api_calls} API calls; reduce selection/questions or edit max_api_calls")
    # tqdm.auto supplies notebook-friendly progress bars for the pending requests.
    from tqdm.auto import tqdm
    for model_id, candidate in READY.items():
        spec, runtime, client = candidate["spec"], None, None
        # Filter jobs to this candidate and the cache retry policy. `continue` skips loading a model
        # when every needed response can already be replayed from cache.
        pending = [j for j in JOBS if j["model_id"] == model_id and should_request(j["cache_key"])]
        if not pending:
            print(model_id, "— using cache")
            continue
        try:
            if spec["kind"] in LOCAL_KINDS:
                runtime = load_local(spec)
            else:
                client = create_api_client(spec)
            # For each job, choose the correct generation function and save the returned status/answer.
            # An unsuccessful API result stops further new calls for that candidate to limit additional spend.
            for job in tqdm(pending, desc=model_id):
                try:
                    system, question = SYSTEMS[job["condition"]], job["question"]["question"]
                    if spec["kind"] in API_KINDS:
                        if api_calls_this_execution >= EXEC["max_api_calls"]:
                            raise RuntimeError("API request cap reached")
                        api_calls_this_execution += 1  # counts attempts, including errors
                    if spec["kind"] in LOCAL_KINDS:
                        result = local_answer(runtime, system, question)
                    elif spec["kind"] == "openai":
                        result = openai_answer(client, spec, system, question)
                    else:
                        result = gemini_answer(client, spec, system, question)
                except Exception as exc:
                    result = safe_error(exc)
                record = save_record(job, candidate, result)
                if record["status"] != "ok" and (spec["kind"] in API_KINDS or record["status"] == "error"):
                    print(model_id, "stopped after", record["status"], record.get("error_type", ""), record.get("finish_reason", ""))
                    break
        except Exception as exc:
            # Setup errors do not turn into baseline answers or zero quality scores.
            note = {"model": model_id, "phase": "load/client setup", **safe_error(exc)}
            run_notes.append(note)
            print(model_id, "could not start:", note["error_type"], "— check candidate paths, API key and device")
        # finally runs whether the try block succeeds, fails, or breaks. Close provider clients,
        # remove runtime references, collect Python objects, and clear unused accelerator allocations.
        finally:
            if client is not None:
                try: client.close()
                except Exception: pass
            runtime = None
            client = None
            gc.collect()
            if spec["kind"] in LOCAL_KINDS:
                import torch
                if torch.backends.mps.is_available(): torch.mps.empty_cache()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("API attempts in this execution:", api_calls_this_execution)
else:
    print("Preview only: no model loaded and no API request sent.")


In [ ]:
# Collect cached/current results for exactly the planned jobs.
# This reads memory only; it does not generate missing responses or turn an absent answer into a score.


# Rebuild the visible result list from current cache entries. Missing requests stay absent;
# only the current bookkeeping question ID is reattached, since the cache key uses question text.
records = []
for job in JOBS:
    record = CACHE.get(job["cache_key"])
    if record is not None:
        # Reattach current question identity: cached text is reusable even if a
        # question's bookkeeping ID changed without changing the actual prompt.
        records.append({**record, "question_id": job["question"]["id"]})
print("Available cached/current responses:", len(records), "of", len(JOBS), "planned requests")


## Score responses and bring in notebook 02's historical baseline

We retain `lab.score_answer` for consistency: correct factual value, hallucination label, over-refusal, and appropriate refusal. It is a simple heuristic, not a semantic judge. Known issues include colour-list punctuation, refusal phrasing, incorrect extra claims, and Arc's unsupported fast charging being represented by a numeric zero. Read answers alongside the scores.

Old notebook 02 responses are re-scored on **exactly the selected questions**, after checking their IDs, text, answerability and expected values. They are labelled **historical**: those files did not save model/source hashes, decoding settings, response status or latency, so they cannot prove an identical historical execution. A freshly evaluated `base` candidate is the stronger comparison.

Primary quality metrics are displayed only when all selected dev questions completed for a model/condition. Error/truncated responses are counted separately, not treated as refusals or scored answers. Custom questions are never included in benchmark scores.

In [ ]:
# Score only valid recorded answers and show historical baseline results with their provenance.
# METRICS names the four displayed quality rates; HISTORY_FILES maps conditions to older saved runs.
# The old run's question text and labels must match selected dev items before its answers are included.
# Historical settings are still unverified, so these results are not proof of an identical earlier experiment.

METRICS = ["accuracy_answerable", "hallucination_rate_answerable", "over_refusal_rate", "appropriate_refusal_rate"]
HISTORY_FILES = {"naive": "01_stock_naive", "engineered": "02_stock_engineered", "with_docs": "03_stock_with_docs"}
historical, history_notes = [], []
for condition in EV["conditions"]:
    path = lab.RUNS / (HISTORY_FILES[condition] + ".json")
    if not path.is_file():
        history_notes.append(f"Missing historical baseline: {path.name}")
        continue
    saved = json.loads(path.read_text())
    answers = saved.get("answers", [])
    # Build an ID lookup. Comparing its length with the answer list catches duplicate IDs;
    # all(...) then requires matching question text, answerability, and expected value for every selected question.
    old = {r["id"]: r for r in answers}
    matches = len(old) == len(answers) and all(
        q["id"] in old and old[q["id"]].get("question") == q["question"]
        and old[q["id"]].get("answerable") == q["answerable"]
        and old[q["id"]].get("expected") == q.get("expected_value") for q in items)
    if not matches:
        history_notes.append(f"{path.name}: question/ground-truth mismatch; not compared")
        continue
    for q in items:
        historical.append({"model_id": "historical_base", "condition": condition, "question_id": q["id"],
                           "question": q["question"], "answer": old[q["id"]]["answer"], "status": "historical",
                           "source_run": path.name})


In [ ]:
# Define scoring and completeness rules without running another model.
# These helpers use the already selected questions and shared scorer, and exclude operational failures.


# Custom questions have no benchmark label and return None. Failed/truncated statuses are excluded.
# lab.score_answer is the unchanged literal-value/refusal heuristic; it does not understand every semantic nuance.
def score_record(record):
    q = items_by_id.get(record["question_id"])
    if q is None or record["status"] not in ("ok", "historical"):
        return None
    return lab.score_answer(record["answer"], q, fidx)

# A condition is complete only when every selected question has one scored response.
# Until then, metric values remain None rather than implying a valid zero or partial-cohort percentage.
def summarise_complete(group):
    scored = [score_record(r) for r in group if score_record(r) is not None]
    complete = len(scored) == len(items) and len({r["question_id"] for r in group if score_record(r) is not None}) == len(items)
    result = {name: None for name in METRICS}
    if complete:
        result.update(lab.summarise(scored))
        # A small selected subset may lack a category. Missing is not zero.
        # A selected subset may contain no answerable or no unanswerable questions.
        # The corresponding rate is undefined, so report None rather than the scorer's denominator-protected zero.
        if not any(q["answerable"] for q in items):
            for name in METRICS[:3]: result[name] = None
        if not any(not q["answerable"] for q in items): result["appropriate_refusal_rate"] = None
    return result, complete, len(scored)


In [ ]:
# Assemble the model-by-condition summary table from the available records.
# A complete cohort is required for headline rates; missing category measurements stay undefined.


# Combine historical and fresh/cached records for display while retaining their model labels.
# Latency is averaged only over responses with measured seconds; historical timings are unknown.
all_records = historical + records
summary_rows = []
for model_id in (["historical_base"] if historical else []) + SELECTED:
    for condition in EV["conditions"]:
        group = [r for r in all_records if r["model_id"] == model_id and r["condition"] == condition and r["question_id"] in items_by_id]
        metrics, complete, n_complete = summarise_complete(group)
        durations = [r["elapsed_seconds"] for r in group if r.get("elapsed_seconds") is not None]
        summary_rows.append({"model": model_id, "condition": condition, "complete": complete,
                             "expected": len(items), "recorded": len(group), "scored": n_complete,
                             "unsuccessful": sum(r["status"] not in ("ok", "historical") for r in group),
                             "mean_seconds": sum(durations)/len(durations) if durations else None, **metrics})
summary_df = pd.DataFrame(summary_rows)
# *METRICS unpacks the metric-name list into the list of displayed columns.
# The completion and unsuccessful-count columns show whether quality percentages are interpretable.
display(summary_df[["model", "condition", "complete", "expected", "scored", "unsuccessful", *METRICS, "mean_seconds"]])
for note in history_notes: print(note)
print("Mean seconds is end-to-end call timing, not a controlled hardware/token-speed benchmark; historical latency is unavailable.")


## Improvement compared with the base, in percentage points

For each complete candidate/condition, use a complete fresh base result if available; otherwise use the checked historical baseline for that condition. Accuracy and appropriate-refusal increases are good; hallucination and over-refusal decreases are good. These are dev results, with no claim of statistical significance.

Example: accuracy moving from 0.40 to 0.60 is **+20 percentage points**. A result from a different condition is never used as the denominator. Small subsets and incomplete responses cannot clear the full development-set bar.

In [ ]:
# Calculate percentage-point changes against a complete baseline under the same prompt condition.
# Prefer a fresh base run; this existing code falls back to historical_base if necessary.
# A historical fallback has unverified execution settings, so its delta remains a historical comparison.
# No new inference or training occurs in this analysis cell.

delta_rows = []
for candidate in summary_rows:
    if candidate["model"] in ("base", "historical_base") or not candidate["complete"]:
        continue
    # next(generator, None) finds the first complete baseline matching the condition,
    # or returns None if none exists. Candidates without a comparable record are skipped.
    baseline = next((r for r in summary_rows if r["model"] == "base" and r["condition"] == candidate["condition"] and r["complete"]), None)
    if baseline is None:
        baseline = next((r for r in summary_rows if r["model"] == "historical_base" and r["condition"] == candidate["condition"] and r["complete"]), None)
    if baseline is None:
        continue
    row = {"model": candidate["model"], "condition": candidate["condition"], "baseline": baseline["model"]}
    for metric in METRICS:
        before, after = baseline.get(metric), candidate.get(metric)
        # Percentage points are 100*(after-before): 0.70 minus 0.60 is +10 pp, not a 10% relative increase.
        # For error rates, a negative change is desirable; for accuracy/refusal success, positive is desirable.
        row[metric + "_change_pp"] = None if before is None or after is None else round(100*(after-before), 2)
    delta_rows.append(row)
if delta_rows:
    display(pd.DataFrame(delta_rows))
else:
    print("No complete trained/API result with a matching baseline yet.")


In [ ]:
# Compare complete full-development results with the saved quality thresholds.
# Each boolean column checks one criterion independently; a small smoke subset is deliberately excluded.


# Assess the predeclared quality bar only when all dev questions were selected and the bar file exists.
# This is still a development check, not a new final-set evaluation.
bar_path = lab.RUNS / "00_bar.json"
if bar_path.is_file() and len(items) == len(all_dev):
    bar = json.loads(bar_path.read_text())["bar"]
    checks = []
    for row in summary_rows:
        if not row["complete"]: continue
        checks.append({"model": row["model"], "condition": row["condition"],
                       "accuracy_pass": row["accuracy_answerable"] >= bar["accuracy_answerable"],
                       "hallucination_pass": row["hallucination_rate_answerable"] <= bar["hallucination_rate_answerable"],
                       "refusal_pass": row["appropriate_refusal_rate"] >= bar["appropriate_refusal_rate"],
                       "over_refusal_pass": row["over_refusal_rate"] <= bar["max_over_refusal"]})
    if checks: display(pd.DataFrame(checks))
else:
    print("Full-dev bar not assessed: use all development questions and an existing notebook01 bar.")


## Inspect the same answer across models

Choose a question ID and condition below. The full answers are shown, including noncompleted responses and the optional custom question. Then inspect improvements and regressions: a higher total score can conceal a newly broken answer or extra unsupported claims.

In [ ]:
# Inspect full responses for one chosen question and find per-question improvements/regressions.
# Choose a question ID and condition from the current selection; 'custom' is available only when configured.
# The table preserves provider status and model version so answer quality is not detached from its execution context.

INSPECT_CONDITION = EV["conditions"][0]
INSPECT_ID = items[0]["id"]  # use "custom" if you set a custom_question

# Look up the requested question, then show every matching model response.
# A custom question is intentionally unscored rather than assigned an invented expected answer.
def inspect_answers(question_id, condition):
    q = next((q for q in questions if q["id"] == question_id), None)
    if q is None: raise ValueError("Choose an ID from the selected questions, or custom")
    print("Question:", q["question"])
    print("Expected:", q.get("expected_value", "unscored custom question"))
    display(pd.DataFrame([
        {"model": r["model_id"], "status": r["status"], "answer": r["answer"],
         "score": score_record(r), "resolved_model": r.get("resolved_model"),
         "finish_reason": r.get("finish_reason")}
        for r in all_records if r["question_id"] == question_id and r["condition"] == condition
    # **{...} expands a dictionary of CSS style properties into named arguments.
    # pre-wrap preserves line breaks, making long answer text easier to inspect without changing it.
    ]).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"}))

inspect_answers(INSPECT_ID, INSPECT_CONDITION)


In [ ]:
# Find question-level correctness changes under the same condition.
# The lookup reuses saved answers; it does not regenerate them or replace a missing baseline response.


changes = []
# Use a tuple (model, condition, question ID) as a dictionary key for quick paired lookup.
# Compare each candidate with fresh base first, then historical base when necessary.
lookup = {(r["model_id"], r["condition"], r["question_id"]): r for r in all_records}
for r in records:
    after = score_record(r)
    if after is None or r["model_id"] == "base": continue
    before_record = lookup.get(("base", r["condition"], r["question_id"]))
    if before_record is None or score_record(before_record) is None:
        before_record = lookup.get(("historical_base", r["condition"], r["question_id"]))
    before = score_record(before_record) if before_record else None
    # Only include cases whose boolean correctness changed. This is a heuristic score transition;
    # read both texts to distinguish real quality change from a scorer limitation.
    if before is not None and before["correct"] != after["correct"]:
        changes.append({"model": r["model_id"], "condition": r["condition"], "question_id": r["question_id"],
                        "change": "improved" if after["correct"] else "regressed",
                        "question": r["question"], "baseline_answer": before_record["answer"], "new_answer": r["answer"]})
if changes:
    display(pd.DataFrame(changes).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"}))
else:
    print("No scored per-question changes available yet.")


## Save this comparison and decide what to try next

The snapshot records config, corpus/evaluation/scorer fingerprints, raw answers, completion coverage and score changes. Each executed comparison gets its own timestamped report; notebook 02's original runs are untouched. Cached API records retain usage and resolved model version. We do not invent currency costs without your actual provider pricing or turn token counts across different tokenizers into an equal-compute claim.

Before concluding that training helped, inspect which facts improved, which regressed, and whether refusals became more appropriate or merely more frequent. Familiar-domain dev improvements are not a final held-out result. No scores are precomputed in this notebook.

In [ ]:
# Save a uniquely named comparison report only when execution was enabled.
# uuid4, from Python's standard uuid module, supplies a random suffix to avoid overwriting earlier reports.
# The UTC timestamp describes when this report was written; hashes link it to its data, scorer and documents.
# This is a report write, not another round of model requests.

if EXEC["run"]:
    from uuid import uuid4
    report_path = CACHE_DIR / (datetime.now(timezone.utc).strftime("comparison_%Y%m%dT%H%M%SZ_") + uuid4().hex[:8] + ".json")
    # Keep configuration, inventory, request plan, errors, metrics, raw responses and historical provenance
    # in one JSON object so a later review can reconstruct what was compared.
    report = {"config": PLAY, "versions": VERSIONS, "question_ids": [q["id"] for q in items],
              "evaluation_sha256": file_sha(lab.EVALS / "dev.json"),
              "facts_sha256": file_sha(lab.RAW / "facts.json"),
              "scorer_sha256": file_sha(ROOT / "src/lab.py"),
              "document_hashes": {p.name: file_sha(p) for p in sorted(lab.RAW.glob("*.md"))},
              "inventory": INVENTORY, "plan": plan, "run_notes": run_notes,
              "api_attempts_this_execution": api_calls_this_execution,
              "summary": summary_rows, "delta_percentage_points": delta_rows,
              "responses": records, "historical_baseline_responses": historical,
              "history_notes": history_notes,
              "notes": "Dev comparison; incomplete quality metrics omitted. Historical settings unverified; shared heuristic scorer has documented limits."}
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    # json.dumps makes readable UTF-8-compatible text; allow_nan=False rejects nonstandard NaN/Infinity.
    # mkdir(..., exist_ok=True) makes the report directory when needed without failing if it already exists.
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
    print("Saved comparison:", report_path)
else:
    print("Preview complete. Edit configs/model_comparison.yaml to select models and enable your run.")
